# Fragmentation Pattern Prediction

분자식(또는 SMILES)을 입력하면, 해당 분자가 이온원에서 **어떤 m/z로 쪼개질 수 있는지** 예측합니다.

### 왜 필요한가?
- MS1 스펙트럼에서 precursor 외에 **In-Source Fragment(ISF)**가 나타남
- 이 fragment의 m/z를 미리 알아야 **정량 시 합산**할 수 있음 (06장)
- 미지 피크의 정체를 **fragmentation 패턴으로 확인** 가능 (09장)

### 예측 방법 
1. **단일 결합 절단 (Bond Cleavage)**: 분자의 모든 non-H 결합을 하나씩 끊어서 가능한 fragment 생성
2. **Neutral Loss**: 알려진 중성 소실 패턴 (H₂O, NO, NO₂, CO 등) 적용
3. **다중 결합 절단**: 2개 이상 결합을 동시에 끊어서 작은 fragment 생성

RDKit을 사용하여 분자 구조를 파싱하고 결합을 절단합니다.

## 환경 설정

In [ ]:
!pip install -q rdkit-pypi pyopenms scipy matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import pyopenms as oms
import os

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw, rdMolDescriptors

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
print('RDKit + pyOpenMS loaded.')

## Step 1. 단일 결합 절단 (Bond Cleavage)

의 핵심 로직:
- 분자의 **모든 결합**을 순회하면서 하나씩 끊음
- H가 관여하는 결합은 건너뜀 (H 이탈은 neutral loss로 처리)
- C-C 결합은 heteroatom에 인접한 경우만 끊음 (순수 탄화수소 결합 보존)
- Ring 결합은 선택적으로 끊음
- 각 절단에서 **두 개의 fragment**가 생성 → 각각의 exact mass 계산

In [ ]:
def fragment_by_bond_cleavage(smiles, break_ring=False, break_cc='hetero'):
    """.BreakBonds port
    
    Parameters
    ----------
    smiles : str - SMILES string
    break_ring : bool - ring bond 절단 여부
    break_cc : 'yes'|'no'|'hetero' - C-C bond 절단 정책
    
    Returns
    -------
    list of dict: fragment info (smiles, mass, formula)
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []
    mol = Chem.AddHs(mol)
    parent_mass = Descriptors.ExactMolWt(Chem.RemoveHs(mol))
    
    fragments = []
    seen = set
    
    for bond in mol.GetBonds:
        a1 = bond.GetBeginAtom
        a2 = bond.GetEndAtom
        
        # Skip H bonds
        if a1.GetSymbol == 'H' or a2.GetSymbol == 'H':
            continue
        
        # C-C bond policy
        if a1.GetSymbol == 'C' and a2.GetSymbol == 'C':
            if break_cc == 'no':
                continue
            elif break_cc == 'hetero':
                # Only break if bonded to heteroatom
                has_hetero = False
                for nbr in list(a1.GetNeighbors) + list(a2.GetNeighbors):
                    if nbr.GetSymbol not in ('C', 'H'):
                        has_hetero = True; break
                if not has_hetero:
                    continue
        
        # Ring bond check
        if bond.IsInRing and not break_ring:
            continue
        
        # Break bond → two fragments
        bond_idx = bond.GetIdx
        try:
            frags = Chem.FragmentOnBonds(mol, [bond_idx], addDummies=False)
            frag_mols = Chem.GetMolFrags(frags, asMols=True, sanitizeFrags=True)
        except:
            continue
        
        for frag_mol in frag_mols:
            frag_mol = Chem.RemoveHs(frag_mol)
            smi = Chem.MolToSmiles(frag_mol)
            if smi in seen or smi == '' or smi == smiles:
                continue
            seen.add(smi)
            mass = Descriptors.ExactMolWt(frag_mol)
            formula = rdMolDescriptors.CalcMolFormula(frag_mol)
            if mass > parent_mass * 0.05:  # >5% of parent
                fragments.append({
                    'smiles': smi, 'formula': formula,
                    'mass': mass,
                    'mz_neg': abs(mass - 1.00728),  # [M-H]-
                    'mz_pos': mass + 1.00728,       # [M+H]+
                    'type': 'bond_cleavage',
                    'bond': f'{a1.GetSymbol}-{a2.GetSymbol}',
                })
    
    fragments.sort(key=lambda x: x['mass'], reverse=True)
    return fragments

# TNT test
tnt_smiles = 'Cc1c(cc(cc1[N+](=O)[O-])[N+](=O)[O-])[N+](=O)[O-]'
tnt_frags = fragment_by_bond_cleavage(tnt_smiles, break_ring=True, break_cc='hetero')

parent_mass = Descriptors.ExactMolWt(Chem.RemoveHs(Chem.AddHs(Chem.MolFromSmiles(tnt_smiles))))
print(f'TNT (C7H5N3O6): parent MW = {parent_mass:.4f}')
print(f'Bond cleavage fragments: {len(tnt_frags)}')
print(f'{"Formula":>15s} {"Mass":>10s} {"[M-H]-":>10s} {"Bond":>6s} {"SMILES"}')
print('-' * 65)
for f in tnt_frags[:15]:
    print(f'{f["formula"]:>15s} {f["mass"]:10.4f} {f["mz_neg"]:10.4f} {f["bond"]:>6s} {f["smiles"]}')

## Step 2. Neutral Loss 예측

분자 전체에서 **작은 중성 분자가 떨어져 나가는** 패턴입니다.
구조를 몰라도 **분자식만으로** 예측 가능합니다.

| Loss | Mass (Da) | 조건 |
|------|-----------|------|
| H₂O | 18.011 | O 포함 |
| CO | 27.995 | C+O 포함 |
| HCN | 27.011 | C+N 포함 |
| NO | 29.997 | N+O 포함 |
| NO₂ | 45.993 | N+2O 포함 |
| CO₂ | 43.990 | C+2O 포함 |

In [ ]:
NEUTRAL_LOSSES = [
    ('H2O',  18.0106, 'O'),
    ('CO',   27.9949, 'CO'),
    ('HCN',  27.0109, 'CN'),
    ('NO',   29.9974, 'NO'),
    ('NO2',  45.9929, 'NO2'),
    ('CO2',  43.9898, 'CO2'),
    ('CH2O', 30.0106, 'CO'),
]

def predict_neutral_losses(parent_mz, formula=None):
    """Neutral loss fragments from parent m/z"""
    results = []
    for name, loss_mass, _ in NEUTRAL_LOSSES:
        frag_mz = parent_mz - loss_mass
        if frag_mz > 50:
            results.append({
                'loss': name, 'loss_mass': loss_mass,
                'frag_mz': frag_mz, 'type': 'neutral_loss',
            })
    return results

# TNT neutral losses from [M-H]- = 226.009
tnt_nl = predict_neutral_losses(226.009)
print(f'TNT [M-H]- neutral losses:')
for nl in tnt_nl:
    print(f'  -{nl["loss"]:6s} ({nl["loss_mass"]:7.4f} Da) -> m/z {nl["frag_mz"]:.4f}')

## Step 3. 통합 예측: Bond Cleavage + Neutral Loss

두 방법을 합쳐서 **전체 예상 fragment 목록**을 생성합니다.
이 목록이 실측 스펙트럼과 대조할 **참조 라이브러리** 역할을 합니다.

In [ ]:
def predict_all_fragments(smiles, adduct_mz, polarity='neg'):
    """Bond cleavage + Neutral loss → combined fragment list"""
    all_frags = []
    
    # Bond cleavage
    bc_frags = fragment_by_bond_cleavage(smiles, break_ring=True, break_cc='hetero')
    for f in bc_frags:
        f['pred_mz'] = f['mz_neg'] if polarity == 'neg' else f['mz_pos']
        all_frags.append(f)
    
    # Neutral loss
    nl_frags = predict_neutral_losses(adduct_mz)
    for f in nl_frags:
        f['pred_mz'] = f['frag_mz']
        f['formula'] = f'-{f["loss"]}'
        all_frags.append(f)
    
    # Remove duplicates (within 0.01 Da)
    unique = []
    seen_mz = []
    for f in all_frags:
        if not any(abs(f['pred_mz'] - s) < 0.01 for s in seen_mz):
            unique.append(f)
            seen_mz.append(f['pred_mz'])
    
    unique.sort(key=lambda x: x['pred_mz'], reverse=True)
    return unique

# TNT combined
tnt_all = predict_all_fragments(tnt_smiles, 226.009, 'neg')
print(f'TNT total predicted fragments: {len(tnt_all)}')
print(f'{"Type":>15s} {"Formula/Loss":>15s} {"Pred m/z":>10s}')
print('-' * 45)
for f in tnt_all:
    print(f'{f["type"]:>15s} {f.get("formula",f.get("loss","")):>15s} {f["pred_mz"]:10.4f}')

## Step 4. 실측 스펙트럼에서 검증

예측된 fragment m/z가 **실제 MS1 스펙트럼에 존재하는지** 확인합니다.
- 매칭된 fragment → 해당 분자의 ISF로 확인
- 06장 Integration에서 이 fragment들의 면적을 합산할 수 있음

In [ ]:
# Load spectrum
exp = oms.MSExperiment
oms.MzMLFile.load('echo-tof-colab/data/mzml/20260330_TOFMS.mzML', exp)

# TIC peak spectrum
best_spec = max((s for s in exp if s.getMSLevel == 1),
                key=lambda s: sum(s.get_peaks[1]))
mz_data, int_data = best_spec.get_peaks

# Match predicted fragments
def match_fragments(pred_frags, mz_arr, int_arr, tol_ppm=20):
    matched, unmatched = [], []
    for f in pred_frags:
        pmz = f['pred_mz']
        tol = pmz * tol_ppm * 1e-6
        mask = np.abs(mz_arr - pmz) < tol
        if mask.any:
            best = np.argmax(int_arr[mask])
            f['obs_mz'] = float(mz_arr[mask][best])
            f['obs_int'] = float(int_arr[mask][best])
            f['ppm'] = (f['obs_mz'] - pmz) / pmz * 1e6
            matched.append(f)
        else:
            unmatched.append(f)
    return matched, unmatched

matched, unmatched = match_fragments(tnt_all, mz_data, int_data)

print(f'Matched: {len(matched)}/{len(tnt_all)} fragments')
print
print(f'{"Type":>15s} {"Formula":>12s} {"Pred m/z":>10s} {"Obs m/z":>10s} {"ppm":>7s} {"Intensity":>12s}')
print('-' * 72)
for f in sorted(matched, key=lambda x: x['obs_int'], reverse=True):
    print(f'{f["type"]:>15s} {f.get("formula",""):>12s} {f["pred_mz"]:10.4f} {f["obs_mz"]:10.4f} {f["ppm"]:+6.1f} {f["obs_int"]:12.1f}')

## Step 5. 시각화: 예측 vs 실측

실측 스펙트럼 위에 예측 fragment 위치를 표시합니다.
- 초록: 매칭됨 (실제 존재)
- 빨강: 미매칭 (예측만, 실측 없음)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

# Spectrum
ax.vlines(mz_data, 0, int_data, colors='navy', linewidth=0.3, alpha=0.3)

# Matched fragments (green)
for f in matched:
    ax.axvline(f['obs_mz'], color='green', linewidth=1.5, alpha=0.7)
    ax.text(f['obs_mz'], f['obs_int'] * 1.05,
            f'{f.get("formula", f.get("loss", ""))}\n{f["obs_mz"]:.2f}',
            ha='center', fontsize=7, color='green', rotation=45)

# Unmatched (red dashes)
ymax = int_data.max
for f in unmatched:
    ax.axvline(f['pred_mz'], color='red', linewidth=0.8, alpha=0.3, linestyle='--')

# Parent
ax.axvline(226.009, color='blue', linewidth=2, alpha=0.8)
ax.text(226.009, ymax * 0.95, 'TNT [M-H]-\n226.009', ha='center', fontsize=9, color='blue')

ax.set_xlabel('m/z')
ax.set_ylabel('Intensity')
ax.set_title(f'TNT Fragmentation: {len(matched)} matched / {len(tnt_all)} predicted')
ax.set_xlim(50, 300)
ax.grid(alpha=0.3)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='green', label=f'Matched ({len(matched)})'),
                    Patch(color='red', alpha=0.3, label=f'Predicted only ({len(unmatched)})'),
                    Patch(color='blue', label='Precursor')], loc='upper right')
plt.tight_layout
plt.show

## Step 6. 다중 화합물 Fragmentation 예측

모든 타겟 화합물에 대해 fragmentation을 예측하고 실측과 대조합니다.

In [ ]:
COMPOUNDS = {
    'TNT':     {'smiles': 'Cc1c(cc(cc1[N+](=O)[O-])[N+](=O)[O-])[N+](=O)[O-]', 'mz': 226.009},
    '2,4-DNT': {'smiles': 'Cc1ccc(cc1[N+](=O)[O-])[N+](=O)[O-]', 'mz': 181.0124},
    'Tetryl':  {'smiles': 'CN(c1c(cc(cc1[N+](=O)[O-])[N+](=O)[O-])[N+](=O)[O-])[N+](=O)[O-]', 'mz': 241.0204},
    'RDX':     {'smiles': 'C1N(CN(CN1[N+](=O)[O-])[N+](=O)[O-])[N+](=O)[O-]', 'mz': 281.048},
}

print(f'{"Compound":10s} {"Predicted":>10s} {"Matched":>8s} {"Top Fragment":>15s} {"Intensity":>12s}')
print('-' * 60)

all_results = {}
for name, info in COMPOUNDS.items:
    frags = predict_all_fragments(info['smiles'], info['mz'], 'neg')
    m, u = match_fragments(frags, mz_data, int_data)
    all_results[name] = {'matched': m, 'unmatched': u, 'total': frags}
    
    top = max(m, key=lambda x: x['obs_int']) if m else None
    top_info = f'{top.get("formula",""):>15s} {top["obs_int"]:12.1f}' if top else f'{"N/A":>15s} {0:12.1f}'
    print(f'{name:10s} {len(frags):10d} {len(m):8d} {top_info}')

## Summary

| 방법 | 입력 | 출력 | 장점 |
|------|------|------|------|
| Bond Cleavage | SMILES | fragment 구조 + mass | 구조 기반, 정확 |
| Neutral Loss | parent m/z | loss pattern | 구조 불필요, 범용 |

### 다음 단계
- **06장 Integration**: 여기서 확인된 ISF fragment의 면적을 precursor와 합산
- **09장 Formula ID**: 미지 피크에 대해 fragmentation 패턴으로 동정

---

⬅️ **Prev:** [MS/MS & Fragment](04%20MSMS%20Fragment.ipynb) | **Next:** [Summation Integration](06%20Summation%20Integration.ipynb) ➡️